In [1]:
import torch
from torch.optim import Adam
from transformers import BertForMaskedLM
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score

1. Get dataset
2. Initialize preprocessing
3. Define hyperparameters
4. Initialize model and optimizer
5. Write training loop
    - Grab batch of data
    - Forward pass in model
    - Compute loss
    - Backpropogation
    - Update parameters

In [2]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

device

'cuda'

In [3]:
import pickle

with open('./data/mimic_icd_data_dict.pkl', 'rb') as f:
    datadict = pickle.load(f)

In [4]:
class MimicIcdDataset(Dataset):
    def __init__(self, data_dict, tokenizer, tokenization_params, max_token_type_ids=100):
        """
        Setup parameters for the dataset
        Parameters
        ----------
        data_dict : dict with keys as patient ids
            training/validation data
        tokenizer : transformers tokenizer
            Tokenizer to do encoding of text
        tokenization_params : dict
            Parameters for tokenization passed to tokenizer. Note, do not set {'return_tensors': 'pt'} because this adds
            an extra dimension. Keep 'return_tensors' to default as this function does the torch tensor conversion when
            it returns the item.
            Suggested parameters: {'max_length': 512, 'truncation': True, 'padding': 'max_length'}
        """
        super().__init__()
        self.data = data_dict
        self.tokenizer = tokenizer
        self.params = tokenization_params
        self.patients = list(self.data.keys())
        self.len = len(self.data)
        self.max_token_type_ids = max_token_type_ids

    def __len__(self):
        return self.len

    def __getitem__(self, index):
        """Processing when returning an item from the Dataset. Converts text into encoding using tokenizer.
        Converts items from encoding to PyTorch tensors
        Parameters
        ----------
        index : int
            index of item to access
        Returns
        -------
        dict
            Output encoding from the tokenizer on the text.
            Relevant keys are ['input_ids', 'attention_mask', 'token_type_ids', 'label']
            What keys are present may depend on tokenizer parameters
        """
        k = self.patients[index]
        data = self.data[k]
        codes = data['icd_code']
        # Get coding embedding ids
        encoding = self.tokenizer(codes, **self.params)
        # Add token type ids for visit sequence
        num_codes_to_keep = min(self.params['max_length'] - 2, len(codes))
        encoding['token_type_ids'][1:num_codes_to_keep+1] = data['visit_order'][:num_codes_to_keep]
        encoding = {key: torch.as_tensor(val) for key, val in encoding.items()}
        return encoding


In [5]:
tokenization_params = {
    'max_length': 128,
    'truncation': True,
    'padding': 'max_length',
    'is_split_into_words': True,
    'return_special_tokens_mask': True
}

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizer-icd-final')

In [7]:
dataset = MimicIcdDataset(datadict, tokenizer, tokenization_params)

In [8]:
BATCH_SIZE = 32

LEARNING_RATE = 1e-4

EPOCHS = 5

In [9]:
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, shuffle=True, batch_size=BATCH_SIZE,
    collate_fn=DataCollatorForLanguageModeling(tokenizer),
    num_workers=4, pin_memory=True)

In [10]:
from transformers import BertConfig

DROPOUT_PROB = 0

config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    # hidden_size=512,
    # num_hidden_layers=6,
    # num_attention_heads=8,
    # intermediate_size=2048,
    hidden_dropout_prob=DROPOUT_PROB,
    attention_probs_dropout_prob=DROPOUT_PROB,
    max_position_embeddings=tokenization_params['max_length'],
    type_vocab_size=100,
    initializer_range=0.01
)

In [11]:
model = BertForMaskedLM(config)
model.to(device)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(26148, 768, padding_idx=0)
      (position_embeddings): Embedding(128, 768)
      (token_type_embeddings): Embedding(100, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True

In [12]:
optimizer = Adam(params=model.parameters(), lr=LEARNING_RATE)

In [13]:
from transformers import get_cosine_schedule_with_warmup

lr_scheduler = get_cosine_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=6000, num_training_steps=30_000)

In [14]:
model.train()

loss_function = torch.nn.CrossEntropyLoss()

for e in range(EPOCHS):
    for idx, batch in tqdm(enumerate(dataloader)):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()

        # B, N, V
        outputs = model(ids, mask, token_type_ids)
        
        # CrossEntropy expects (B, V, d1), so permute dims
        logits = outputs.logits.permute(0, 2, 1)
        loss = loss_function(logits, labels)
        loss.backward()
        
        optimizer.step()
        lr_scheduler.step()

        if idx % 100 == 0:
            # Get actual prediction
            preds = torch.argmax(logits, axis=1)
            # Get active labels
            active_mask = labels.view(-1) != -100
            active_labels = torch.masked_select(labels.view(-1), active_mask)
            active_preds = torch.masked_select(preds.view(-1), active_mask)
            acc = accuracy_score(active_preds.cpu().numpy(), active_labels.cpu().numpy())
            print(f'epoch: {e}, batch: {idx}, loss: {loss.item():.4f}, accuracy: {acc:.2f}')
    torch.save(model, f'./data/models/bert_for_icd_epoch_{e}.pt')


1it [00:01,  1.00s/it]

epoch: 0, batch: 0, loss: 10.2262, accuracy: 0.00


101it [00:53,  1.91it/s]

epoch: 0, batch: 100, loss: 10.1113, accuracy: 0.00


201it [01:45,  1.87it/s]

...

5801it [49:02,  1.95it/s]

epoch: 4, batch: 5800, loss: 3.6065, accuracy: 0.33


5901it [49:53,  1.98it/s]

epoch: 4, batch: 5900, loss: 4.1871, accuracy: 0.29


5944it [50:15,  1.97it/s]


GPU can handle a batch up to 128 wrt memory and processing time. However, current processing code is too slow on the CPU to get a batch ready before the GPU can run it. 

In [15]:
# Check acc and loss on last batch
loss.item()

5.96185827255249

In [16]:
acc

0.2920353982300885